In [1]:
!pip install -q segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 56.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
import time

import segmentation_models_pytorch as smp
from torch import nn
from torch.optim import AdamW
import torch
from sklearn.model_selection import train_test_split
import wandb
from torch.utils.data import Dataset, DataLoader, Subset
import albumentations as A
import torch

seed = 42


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = seed


class CFG:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    loss_fn = smp.losses.SoftBCEWithLogitsLoss()
    num_epochs = 5
    bs = 32
    use_wandb = True
    encoder = "resnet34"


if CFG.use_wandb:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("wandb")
    wandb.login(key=secret_value_0)
    wandb.init(
        project="ship-detection",
        config={k: v for k, v in dict(vars(CFG)).items() if "__" not in k},
        group="debug",
    )
else:
    from dotenv import load_dotenv

    load_dotenv()
    wandb.login()
    wandb.init(
        project="ship-detection",
        config={k: v for k, v in dict(vars(CFG)).items() if "__" not in k},
        group="debug",
    )


def get_mask(labels):
    """create mask with the help of list of encoded pixels"""
    mask = np.zeros(768 * 768)
    for label in labels:
        label = label.split()
        start = list(map(int, label[::2]))
        run_len = list(map(int, label[1::2]))

        for s, r in zip(start, run_len):
            mask[s: s + r] = 1
    mask = mask.reshape(768, 768).T
    return mask


tfms = A.Compose([A.Resize(height=224, width=224), A.Normalize()])


class ShipData(Dataset):
    def __init__(self, root_dir, train_df, transform=None):
        self.root_dir = root_dir
        self.image_ids = train_df["ImageId"].unique()
        self.tfms = transform
        self.df = train_df

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        labels = self.df.loc[self.df["ImageId"] == img_id]["EncodedPixels"]
    
        try:
            img_path = os.path.join(self.root_dir / "train_v2", img_id)
            img = Image.open(img_path).convert("RGB")
            img = np.array(img)
        except Exception as e:
            print(f"⚠️ Skipping corrupted image {img_id}: {e}")
            return self.__getitem__((idx + 1) % len(self.image_ids))  # pick another image
    
        mask = get_mask(labels)
        if self.tfms:
            result = self.tfms(image=img, mask=mask)
            img, mask = result["image"], result["mask"]
        img = torch.tensor(img).permute(2, 0, 1)
        mask = torch.tensor(mask)
        return img, mask


def dice_score(yb, yb_pred, th=0.5):
    yb = yb.flatten()
    yb_pred = yb_pred.flatten() > th

    total_pixel_matched = (yb == yb_pred).sum()

    dice = (2 * total_pixel_matched) / (2 * len(yb))
    return dice


def train_one_epoch(dl, model, optimizer, loss_fn):
    running_loss = 0
    running_acc = 0
    for xb, yb in tqdm(dl):
        xb = xb.to(CFG.device)
        yb = yb.to(CFG.device)

        logit = model(xb)
        loss = loss_fn(logit.squeeze(), yb)
        acc = dice_score(yb, logit)

        running_loss += loss.item()
        running_acc += acc.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return running_loss / len(dl), running_acc / len(dl)


@torch.no_grad()
def valid_one_epoch(dl, model, optimizer, loss_fn):
    running_loss = 0
    running_acc = 0
    for xb, yb in tqdm(dl):
        xb = xb.to(CFG.device)
        yb = yb.to(CFG.device)

        logit = model(xb)
        loss = loss_fn(logit.squeeze(), yb)

        acc = dice_score(yb, logit)

        running_loss += loss.item()
        running_acc += acc.item()
    return running_loss / len(dl), running_acc / len(dl)


path = Path("/kaggle/input/airbus-ship-detection")
df = pd.read_csv(path / "train_ship_segmentations_v2.csv")
df.fillna("", inplace=True)

train_df, valid_df = train_test_split(df, test_size=0.2)
train_ds = ShipData(path, train_df, transform=tfms)
valid_ds = ShipData(path, valid_df, transform=tfms)

train_dl = DataLoader(
    train_ds, batch_size=CFG.bs, num_workers=2, pin_memory=True, shuffle=True
)
valid_dl = DataLoader(
    valid_ds, batch_size=CFG.bs, num_workers=2, pin_memory=True, shuffle=False
)


# plot a batch with images and masks for total image of 16
def visualize_batch(xb, yb):
    xb, yb = next(iter(train_dl))

    num_images = min(16, xb.size(0))  # handle smaller batches
    cols = 4
    rows = (num_images + cols - 1) // cols

    plt.figure(figsize=(12, 12))

    for i in range(num_images):
        plt.subplot(rows, cols, i + 1)
        img = xb[i].permute(1, 2, 0).numpy()
        mask = yb[i].numpy()
        plt.imshow(img)
        plt.imshow(mask, alpha=0.5, cmap="gray")
        plt.axis("off")

    plt.tight_layout()
    plt.savefig("images.png")


model = smp.Unet(
    encoder_name=CFG.encoder,
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)
model.to(CFG.device)
lr = 0.001
optimizer = AdamW(lr=lr, params=model.parameters())

print(f"train_dl {len(train_dl)}")


for epoch in range(CFG.num_epochs):
    tik = time.time()
    train_loss, train_dice = train_one_epoch(train_dl, model, optimizer, CFG.loss_fn)
    valid_loss, valid_dice = valid_one_epoch(valid_dl, model, optimizer, CFG.loss_fn)
    if CFG.use_wandb:
        wandb.log(
            {
                "Train Loss": train_loss,
                "Valid Loss": valid_loss,
                "train dice": train_dice,
                "Valid Dice": valid_dice,
                "LR": lr,
            }
        )
    tok = time.time()
    print(
        f"ecoch {epoch} | train_loss {train_loss:.4f} | train_dice {train_dice:.4f} | valid_loss {valid_loss:.4f} | valid_dice {valid_dice:.4f} | time {tok-tik:.2f}s"
    )


torch.save(model.state_dict(), "model.pth")
if CFG.use_wandb:
    wandb.save("model.pth")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

train_dl 4902


 67%|██████▋   | 3276/4902 [45:24<19:29,  1.39it/s]

...

ecoch 3 | train_loss 0.0014 | train_dice 0.9995 | valid_loss 0.0021 | valid_dice 0.9991 | time 3996.51s


 36%|███▌      | 1742/4902 [20:32<35:11,  1.50it/s]

⚠️ Skipping corrupted image 6384c3e78.jpg: image file is truncated (51 bytes not processed)


100%|██████████| 1347/1347 [09:12<00:00,  2.44it/s]


ecoch 4 | train_loss 0.0013 | train_dice 0.9995 | valid_loss 0.0025 | valid_dice 0.9991 | time 4073.97s
